# Run and replay search

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Xmaster6y/lczerolens/blob/main/docs/source/notebooks/features/replayable-search.ipynb)

`ReferenceSearch` is a deterministic, sequential evidence producer. It is useful for testing analysis and replay logic, but it is not algorithmically equivalent to the official lc0 engine. This offline demo uses the versioned fixture rather than trained weights.

In [ ]:
# Colab starts from a clean runtime; local and docs builds skip this setup.
import importlib.util
import os
from pathlib import Path
import subprocess
import sys

if importlib.util.find_spec("google.colab") is not None:
    checkout = Path("/content/lczerolens")
    if not checkout.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/Xmaster6y/lczerolens.git", str(checkout)],
            check=True,
        )
    else:
        subprocess.run(["git", "-C", str(checkout), "pull", "--ff-only"], check=True)
    os.chdir(checkout)
    sys.path.insert(0, str(checkout))
    sys.path.insert(0, str(checkout / "src"))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

In [ ]:
import chess

from examples.decision_analysis_tutorial import load_fixture_evaluator
from lczerolens import (
    ReferenceSearch,
    Simulations,
    replay_retained_events,
    replay_search_trace,
)
from lczerolens.search.trace import SearchCapability

board = chess.Board()
runtime = load_fixture_evaluator()
result = ReferenceSearch(runtime.evaluator, c_puct=1.0).run(board, Simulations(4))
(result.move.uci(), result.capability, len(result.trace.events))

Capability checks are part of the contract. A replayable reference trace exposes full simulation events; an official-engine trace may expose only root snapshots, and consumers must not infer missing events.

In [ ]:
result.trace.require(SearchCapability.REPLAYABLE)
root_rows = [
    {"move": move, "prior": round(action.prior, 4), "visits": action.visits}
    for move, action in result.root.items()
    if action.visits
]
root_rows

Semantic replay reconstructs the deterministic tree from recorded events instead of trusting the recorded final snapshot. Retained-event replay answers a different question: what root decision is represented by an explicit subset of the original simulations?

In [ ]:
semantic = replay_search_trace(result.trace)
retained = replay_retained_events(result.trace, ("simulation-0", "simulation-2"))
assert semantic.selected_move == result.move.uci()
{
    "semantic_move": semantic.selected_move,
    "retained_move": retained.selected_move,
    "retained_events": retained.plan.retained_event_ids,
    "observed_costs": retained.costs,
}

## Use the official engine boundary

For production lc0 output, construct `LczeroSearch(executable=..., network=..., engine_version=...)` and run it with `Nodes(...)` or `Time(...)`. That process-backed adapter is intentionally not executed in this hermetic notebook. It records only public root evidence and advertises at most root-snapshot capability; use the opt-in pinned live conformance test to validate a particular binary and network.